## The Blob in CESM2-FOSI: Subsurface, heat budget, MLD

## Imports

In [12]:
import cftime
import xesmf as xe
import pop_tools
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

## Functions

In [2]:
def get_var_paths(directory, var):
    prefixes_to_match_fut = ['b.e21.BSSP370cmip6.', 'b.e21.BSSP370smbb.']
    prefixes_to_match_hist = ['b.e21.BHISTcmip6.', 'b.e21.BHISTsmbb.']
    prefixes_fut, prefixes_hist = [], []
    for filename in os.listdir(directory):
        if any(filename.startswith(p) for p in prefixes_to_match_fut) and filename.endswith('.nc'):
            prefixes_fut.append(filename.rsplit('.', 3)[0])
        if any(filename.startswith(p) for p in prefixes_to_match_hist) and filename.endswith('.nc'):
            prefixes_hist.append(filename.rsplit('.', 3)[0])
    return sorted(set(prefixes_hist)), sorted(set(prefixes_fut))

In [3]:
def detrend_term_fosi(da):
    term_nan = xr.where(da == 0., np.nan, da) * NO_SEC_IN_1_MON
    mean, trend, seasonal, notrend = afuncs.calculate_pop_grid_anomalies_trend_features(term_nan)
    return notrend

In [4]:
def find_identifier_with_index(prefixes, identifier):
    return [(p, i) for i, p in enumerate(prefixes) if identifier in p]

In [5]:
def get_hist_file_paths(var, directory, path_intermed_hist, index):
    attrib_title = path_intermed_hist[index]
    file_paths = [f'{directory}{attrib_title}.{var}.{y}01-{y+9}12.nc' for y in range(1850, 2010, 10)]
    file_paths.append(f'{directory}{attrib_title}.{var}.201001-201412.nc')
    return file_paths

In [6]:
def get_fut_file_paths(var, directory, path_intermed_fut, index):
    attrib_title = path_intermed_fut[index]
    file_paths = [f'{directory}{attrib_title}.{var}.{y}01-{y+9}12.nc' for y in range(2015, 2095, 10)]
    file_paths.append(f'{directory}{attrib_title}.{var}.209501-210012.nc')
    return file_paths

In [7]:
def file_path_to_var_ds(file_paths):
    return xr.open_mfdataset(file_paths, concat_dim='time', combine='nested', parallel=True)

In [8]:
def get_ds_var(directory, var, comp, index_hist):
    path_intermed_hist, path_intermed_fut = get_var_paths(directory, var)
    filename_identifier = '.'.join(path_intermed_hist[index_hist].rsplit('.', 5)[1:4])
    index_fut = find_identifier_with_index(path_intermed_hist, filename_identifier)[0][1]
    hist_file_paths = get_hist_file_paths(var, directory, path_intermed_hist, index_hist)
    fut_file_paths = get_fut_file_paths(var, directory, path_intermed_fut, index_fut)
    return file_path_to_var_ds(hist_file_paths), file_path_to_var_ds(fut_file_paths)

In [9]:
def regrid_SMYLE(ds, glat=1, glon=1):
    ds = ds.rename({'TLONG': 'lon', 'TLAT': 'lat'})
    ds_out = xe.util.grid_global(glon, glat)
    regridder = xe.Regridder(ds, ds_out, 'bilinear', periodic=True)
    regridded = regridder(ds)
    new_coords = regridded.assign_coords({'y': regridded.lat[:, 0].values, 'x': regridded.lon[0].values})
    return new_coords.drop_vars(['lat', 'lon']).rename({'x': 'lon', 'y': 'lat'})

## Load data

In [10]:
# --- Build CESMLENS_SST reference grid ---
var, comp = 'SST', 'atm'
directory = f'/glade/campaign/cgd/cesm/CESM2-LE/{comp}/proc/tseries/month_1/{var}/'
ds_var_hist_SST, ds_var_fut_SST = get_ds_var(directory, 'SST', 'atm', 0)
CESMLENS_SST = xr.concat([
    ds_var_hist_SST.SST.sel(time=slice('1979-01-01', '2015-01-01')),
    ds_var_fut_SST.SST.sel(time=slice('2015-02-01', '2020-12-01'))
], dim='time').compute()

In [11]:
# --- Canonical mask + FOSI Blob labels ---
mask_3 = xr.open_dataset('mean_mask_3.nc').mhw_obj
mhw_obj_mask = xr.where(mask_3 > 0.1, 1, 0)
fosi_blobs_full = xr.open_dataset('fosi_blobs_r2.nc')
object_id = 94.0
firstyear, lastyear = 1979, 2020

grid = pop_tools.get_grid('POP_gx1v7')
region_mask = xr.where((grid['REGION_MASK'] > 0) & (grid['REGION_MASK'] < 9), 1, np.nan)

## Comparing the fixed footprint against the Blob's time-varying footprint

See other notebooks!

## Subsurface

In [ ]:
# FOSI raw 3D TEMP -> regrid -> detrend -> canonical/time-varying profiles
field = 'TEMP'
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
ds_smyle_fosi_temp_deep = xr.open_dataset(fpath + fname)[field].isel(z_t=slice(0, 23))  # to ~236m
fosi_montime_vals = [cftime.DatetimeNoLeap(1958 + year, 1 + month, 15) for year in range(63) for month in range(12)]
ds_smyle_fosi_temp_deep['time'] = fosi_montime_vals

fosi_anom_1deg_wzeros_temp = regrid_SMYLE(ds_smyle_fosi_temp_deep)
fosi_anom_1deg_temp = fosi_anom_1deg_wzeros_temp.where(fosi_anom_1deg_wzeros_temp != 0, np.nan)
regridder_temp = xe.Regridder(
    fosi_anom_1deg_temp.sel(time=slice('1979-01-01', '2020-12-01')),
    CESMLENS_SST, 'nearest_s2d', periodic=True
)
regridded_temp_deep = regridder_temp(fosi_anom_1deg_temp.sel(time=slice('1979-01-01', '2020-12-01')))

smaller_region_temp_deep = regridded_temp_deep.sel(lat=slice(12, 65), lon=slice(155, 245))
anomalies_deep = calculate_anomalies_trend_features_4d(smaller_region_temp_deep)

smaller_mhw_obj_mask = mhw_obj_mask.sel(lat=slice(12, 65), lon=slice(155, 245))
fosi_blobs_cropped = fosi_blobs_full.labels.sel(lat=slice(12, 65), lon=slice(155, 245))

blob_times = fosi_blobs_cropped.time.where((fosi_blobs_cropped == object_id).any(dim=('lat', 'lon')), drop=True)
full_time_index = anomalies_deep.time
first_idx = int(np.where(full_time_index.values == blob_times.values[0])[0][0])
last_idx = int(np.where(full_time_index.values == blob_times.values[-1])[0][0])
pad_start = max(0, first_idx - 2)
pad_end = min(len(full_time_index) - 1, last_idx + 2)
padded_times = full_time_index.isel(time=slice(pad_start, pad_end + 1))

# --- Canonical profile ---
canonical_masked_deep = anomalies_deep.sel(time=padded_times).where(smaller_mhw_obj_mask == 1)
canonical_profile_deep = canonical_masked_deep.mean(dim=('lat', 'lon')).compute()

# --- Time-varying profile ---
first_mask = (fosi_blobs_cropped.sel(time=blob_times.values[0]) == object_id).reset_coords('time', drop=True)
last_mask = (fosi_blobs_cropped.sel(time=blob_times.values[-1]) == object_id).reset_coords('time', drop=True)

tv_profile_deep_list = []
for t in padded_times.values:
    if t in blob_times.values:
        m = (fosi_blobs_cropped.sel(time=t) == object_id).reset_coords('time', drop=True)
    elif t < blob_times.values[0]:
        m = first_mask
    else:
        m = last_mask
    month_anom = anomalies_deep.sel(time=t).where(m).mean(dim=('lat', 'lon'))
    tv_profile_deep_list.append(month_anom.assign_coords(time=t))
tv_profile_deep = xr.concat(tv_profile_deep_list, dim='time')

/glade/derecho/scratch/cassiacai/tmp/ipykernel_11247/125283344.py:5: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds_smyle_fosi_temp_deep = xr.open_dataset(fpath + fname)[field].isel(z_t=slice(0, 23))  # to ~236m


## Heat budget

In [ ]:
# FOSI heat budget (fixed)
fosi_heat_budget = xr.open_dataset('/glade/derecho/scratch/cassiacai/fosi_heat_budget_155m.nc')
fosi_heat_budget['SUBGRID'] = fosi_heat_budget['TEND_TEMP'] - (
    fosi_heat_budget['TOTADV'] + fosi_heat_budget['SHF_depth'] + fosi_heat_budget['QSW_3D']
)
fosi_hb_detrended = {}
for term in ['TOTADV', 'SHF_depth', 'QSW_3D', 'SUBGRID', 'TEND_TEMP']:
    notrend = detrend_term_fosi(fosi_heat_budget[term])
    key = 'TOT_ADV' if term == 'TOTADV' else term
    fosi_hb_detrended[key] = notrend

fosi_hb_cropped = {term: da.isel(nlat_t=lat_slice, nlon_t=lon_slice) for term, da in fosi_hb_detrended.items()}
fosi_hb_masked = {term: da.where(NEPac_MHW_renamed_latlon == 1) for term, da in fosi_hb_cropped.items()}

fosi_blobs_labels = xr.open_dataset('fosi_blobs_r2.nc').labels
blob_times_hb = fosi_blobs_labels.time.where((fosi_blobs_labels == object_id).any(dim=('lat', 'lon')), drop=True)
padded_times_fosi = build_padded_time_window(fosi_hb_masked['TOT_ADV'].time, blob_times_hb)
fosi_hb_profile = {
    term: da.sel(time=padded_times_fosi).mean(dim=('nlat_t', 'nlon_t')).compute()
    for term, da in fosi_hb_masked.items()
}
fosi_phases_5, fosi_n_phases, fosi_peak_date = phase_split_adaptive(fosi_hb_profile)

## Mixed layer depth

In [ ]:
# FOSI MLD (HMXL), canonical and time-varying
field_mld = 'HMXL'
fname_mld = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field_mld}.030601-036812.nc'
ds_smyle_fosi_hxml = xr.open_dataset(fpath + fname_mld)[field_mld]
ds_smyle_fosi_hxml['time'] = fosi_montime_vals
fosi_mld = ds_smyle_fosi_hxml[
    ds_smyle_fosi_hxml.time['time.year'].isin(list(range(firstyear, lastyear + 1)))
].where(region_mask == 1, np.nan)

fosi_anom_1deg_wzeros_mld = regrid_SMYLE(fosi_mld)
fosi_anom_1deg_mld = fosi_anom_1deg_wzeros_mld.where(fosi_anom_1deg_wzeros_mld != 0, np.nan)
regridder_mld = xe.Regridder(
    fosi_anom_1deg_mld.sel(time=slice('1979-01-01', '2021-01-01')),
    CESMLENS_SST, 'nearest_s2d', periodic=True
)
regridded_mld = regridder_mld(fosi_anom_1deg_mld.sel(time=slice('1979-01-01', '2021-01-01')))
smaller_region_mld = regridded_mld.sel(lat=slice(12, 65), lon=slice(155, 245))

canonical_mld_masked = smaller_region_mld.sel(time=padded_times).where(smaller_mhw_obj_mask == 1)
canonical_mld_profile = (canonical_mld_masked.mean(dim=('lat', 'lon')) / 100).compute()

tv_mld_list = []
for t in padded_times.values:
    if t in blob_times.values:
        m = (fosi_blobs_cropped.sel(time=t) == object_id).reset_coords('time', drop=True)
    elif t < blob_times.values[0]:
        m = first_mask
    else:
        m = last_mask
    month_mld = smaller_region_mld.sel(time=t).where(m).mean(dim=('lat', 'lon')) / 100
    tv_mld_list.append(month_mld.assign_coords(time=t))
tv_mld_profile = xr.concat(tv_mld_list, dim='time')

In [ ]:
# Climatological MLD check (used to justify 155m integration depth)
fosi_mld_full = ds_smyle_fosi_hxml[
    ds_smyle_fosi_hxml.time['time.year'].isin(list(range(firstyear, lastyear + 1)))
] / 100
fosi_mld_cropped = fosi_mld_full.isel(nlat=lat_slice, nlon=lon_slice).rename({'nlat': 'nlat_t', 'nlon': 'nlon_t'})
fosi_mld_masked_clim = fosi_mld_cropped.where(NEPac_MHW_renamed_latlon == 1)
fosi_mld_area_avg = fosi_mld_masked_clim.mean(dim=('nlat_t', 'nlon_t'))
climatology = fosi_mld_area_avg.groupby('time.month').mean(dim='time')
deepest_month_idx = int(climatology.argmax())
print(f"Deepest climatological month MLD: {float(climatology.isel(month=deepest_month_idx)):.1f}m")